# 數列/前綴和
Status: in_use  
Prereq: Python 基本輸入輸出、for 迴圈、if 判斷  
Can-do checklist:  
- [ ] 能區分等差、等比、遞迴定義數列
- [ ] 能用 Python 生成前 N 項與第 N 項
- [ ] 能辨識 APCS 常見出題模式（模擬、前綴和、遞迴轉迴圈）

> 目標：把高中常見數列公式與 APCS 實作串起來，做到「看懂題意 -> 選對模型 -> 寫出可過測資的程式」。


## 題目先看（LeetCode 724: Find Pivot Index）
給定整數陣列 `nums`，請找出最左邊的 pivot index。

pivot index 定義：
- $i$ 左邊所有元素總和等於 $i$ 右邊所有元素總和

補充：
- 若 $i$ 在最左邊，左邊總和視為 $0$
- 若 $i$ 在最右邊，右邊總和視為 $0$
- 若不存在，回傳 $-1$

範例：
- `nums = [1,7,3,6,5,6]` -> `3`
- `nums = [1,2,3]` -> `-1`
- `nums = [2,1,-1]` -> `0`


## 0. intro：先有問題意識，再選工具
以 LeetCode 724（Find Pivot Index）為例，先不要急著寫程式，先問：
- 我要找的條件是什麼？
  - 某個索引 $i$，左側和等於右側和。
- 直覺解法是什麼？
  - 對每個 $i$ 都重算左右和（雙層迴圈，$O(n^2)$）。
- 為什麼需要優化？
  - 當 $n$ 大時，重複加總太慢。
- 可以重用哪些資訊？
  - 全部總和 $\text{total}$、目前左和 $\text{left}$。

核心轉換：
- $\text{right} = \text{total} - \text{left} - \text{nums}[i]$
- 只要一路更新 $\text{left}$，每個位置都能在 $O(1)$ 算出右和。

這就是本份講義的主軸：
1. 先辨識數學/資料關係。
2. 再把關係轉成可重用的狀態（如前綴和、累計量）。
3. 最後才落地成 APCS 可過測資的程式。


## 1. 數列核心觀念速覽
- 數列（sequence）：按順序排出的數字，例如 $a_1, a_2, a_3, \ldots$。
- 通項公式：直接由 $n$ 算 $a_n$，例如 $a_n = 2n + 1$。
- 遞迴公式：用前幾項推出下一項，例如 $a_n = a_{n-1} + 3$。

高中重點：
- 會背與會用等差、等比公式。
- 會算前 $n$ 項和。
- 會判斷數列是否單調、是否有界（延伸）。

APCS重點：
- 題目常不直接說「等差等比」，而是藏在文字敘述裡。
- 常要求「前幾項輸出」、「找第 $k$ 項」、「累加到超過某值」。
- 關鍵是把數學模型轉成迴圈與變數更新。


## 2. 等差數列（Arithmetic Sequence）
定義：相鄰兩項差固定為常數 $d$。
- 通項：$a_n = a_1 + (n-1)d$
- 前 $n$ 項和：$S_n = \dfrac{n(a_1 + a_n)}{2} = \dfrac{n(2a_1 + (n-1)d)}{2}$

常見陷阱：
- $n$ 從 1 開始還是 0 開始。
- 題目給的是第 0 項時，公式要改成 $a_n = a_0 + nd$。


### 練習（等差）
1. 已知 $a_1=4,d=5$，求第 $20$ 項。
2. 已知 $a_1=-3,d=2$，求前 $30$ 項和。
3. 題目給第 $0$ 項：$a_0=7,d=-1$，求第 $n$ 項公式並實作函式。


In [1]:
# TODO: 完成等差練習
# 建議：先寫 arithmetic_nth_0_based(a0, d, n)

def arithmetic_nth_0_based(a0, d, n):
    return a0 + n * d

print(arithmetic_nth_0_based(7, -1, 10))  # 預期 -3


-3


In [2]:
def arithmetic_nth(a1, d, n):
    # n 為第 n 項（n 從 1 開始）
    return a1 + (n - 1) * d


def arithmetic_sum(a1, d, n):
    an = arithmetic_nth(a1, d, n)
    return n * (a1 + an) // 2


a1, d = 5, 3
for n in range(1, 6):
    print(f"n={n}, an={arithmetic_nth(a1, d, n)}")

print("S10 =", arithmetic_sum(5, 3, 10))


n=1, an=5
n=2, an=8
n=3, an=11
n=4, an=14
n=5, an=17
S10 = 185


## 3. 等比數列（Geometric Sequence）
定義：相鄰兩項比固定為常數 $r$。
- 通項：$a_n = a_1 r^{n-1}$
- 前 $n$ 項和（$r \ne 1$）：$S_n = a_1\dfrac{1-r^n}{1-r}$

APCS 實務注意：
- 浮點數可能造成誤差，能用整數就不用浮點。


### 練習（等比）
1. 已知 $a_1=3,r=2$，求第 $8$ 項與前 $8$ 項和。
2. 若 $r=1$，前 $n$ 項和如何簡化？請寫出程式特判。


In [4]:
# TODO: 完成等比練習（含 r=1 特判）

def geometric_sum(a1, r, n):
    if r == 1:
        return a1 * n
    term = a1
    total = 0
    for _ in range(n):
        total += term
        term *= r
    return total

print(geometric_sum(3, 2, 8))
print(geometric_sum(5, 1, 8))


765
40


In [5]:
def geometric_nth(a1, r, n):
    return a1 * (r ** (n - 1))


def geometric_sum_iter(a1, r, n):
    # 用迴圈累加，避免公式在某些情況的浮點誤差
    term = a1
    total = 0
    for _ in range(n):
        total += term
        term *= r
    return total


print("a6 =", geometric_nth(2, 3, 6))
print("S6 =", geometric_sum_iter(2, 3, 6))


a6 = 486
S6 = 728


## 4. 遞迴定義數列：Fibonacci 作為代表
- `F0 = 0, F1 = 1`
- `Fn = F(n-1) + F(n-2)`

這類題目是 APCS 常客，重點不是背公式，而是：
- 先找狀態（需要記住哪些前項）
- 再決定更新順序


### 練習（Fibonacci）
1. 寫 `fib_list(n)` 回傳 $F_0$ 到 $F_n$。
2. 找出你程式在 $n=0,1,2$ 是否都正確。
3. 挑戰：只用兩個變數，不可開陣列。


In [6]:
# TODO: 完成 fib_list

def fib_list(n):
    if n == 0:
        return [0]
    arr = [0, 1]
    for _ in range(2, n + 1):
        arr.append(arr[-1] + arr[-2])
    return arr

print(fib_list(10))


[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55]


In [7]:
def fib_iter(n):
    if n == 0:
        return 0
    if n == 1:
        return 1

    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b


for n in range(11):
    print(n, fib_iter(n))


0 0
1 1
2 1
3 2
4 3
5 5
6 8
7 13
8 21
9 34
10 55


## 5. APCS 常見考點 A：前 N 項輸出 + 格式控制
常見輸出要求：
- 空白分隔
- 逗號分隔
- 不要多一個尾端空白

寫法建議：
- 先把結果放進 list
- 最後用 `' '.join(...)` 一次輸出


### 練習（格式控制）
1. 輸出前 $n$ 項時，不可有尾端空白。
2. 改成逗號分隔（`,`）後再輸出一次。
3. 加一條規則：每 5 個數換行。


In [8]:
# TODO: 改寫輸出格式
arr = [1, 3, 5, 7, 9, 11, 13]
print(' '.join(map(str, arr)))
print(','.join(map(str, arr)))


1 3 5 7 9 11 13
1,3,5,7,9,11,13


In [9]:
def first_n_arithmetic(a1, d, n):
    arr = []
    cur = a1
    for _ in range(n):
        arr.append(cur)
        cur += d
    return arr


seq = first_n_arithmetic(7, -2, 8)
print(" ".join(map(str, seq)))


7 5 3 1 -1 -3 -5 -7


## 6. APCS 常見考點 B：前綴和（Prefix Sum）
若題目要多次查詢「第 l 項到第 r 項和」，前綴和很常用。
- `prefix[i]` = 前 i 項總和（含第 i 項）
- 區間和：`sum(l..r) = prefix[r] - prefix[l-1]`


### 練習（前綴和）
1. 給陣列與 $q$ 次查詢，回傳每個區間和。
2. 試比較暴力法與前綴和法在大量查詢時的差異。
3. 挑戰：改成 0-based 查詢版本。


In [10]:
# TODO: 多查詢版本

def answer_queries(arr, queries):
    prefix = [0]
    for x in arr:
        prefix.append(prefix[-1] + x)
    out = []
    for l, r in queries:  # 1-based
        out.append(prefix[r] - prefix[l - 1])
    return out

print(answer_queries([3,1,4,1,5,9], [(1,3), (2,5), (4,6)]))


[8, 11, 15]


In [11]:
def build_prefix(arr):
    prefix = [0]
    for x in arr:
        prefix.append(prefix[-1] + x)
    return prefix


def range_sum(prefix, l, r):
    # l, r 為 1-based index
    return prefix[r] - prefix[l - 1]


arr = [3, 1, 4, 1, 5, 9]
prefix = build_prefix(arr)
print("prefix:", prefix)
print("sum(2..5)=", range_sum(prefix, 2, 5))


prefix: [0, 3, 4, 8, 9, 14, 23]
sum(2..5)= 11


## 7. APCS 常見考點 C：模擬到達門檻
題型：給定一個數列規則，問最小的 `k` 使得前 k 項和 >= X。
這類題目常用 while/for 模擬，不一定有封閉公式。


### 練習（門檻模擬）
1. 等差情境：最小 $k$ 使前 $k$ 項和 $\ge X$。
2. 每次增加量不是常數，而是前一次的 2 倍，請模擬。
3. 思考：何時可以不用模擬，直接用數學式反推？


In [12]:
def min_k_for_target(a1, d, target):
    total = 0
    cur = a1
    k = 0
    while total < target:
        total += cur
        cur += d
        k += 1
    return k, total


k, s = min_k_for_target(2, 3, 100)
print("k=", k, "sum=", s)


k= 8 sum= 100


## 8. 高中觀念連結：等差 vs 等比怎麼快速判斷
給前幾項時：
- 差固定 -> 等差
- 比固定（且前項不為 0） -> 等比
- 都不是 -> 可能是遞迴或分段規則


### 練習（判斷題型）
判斷下列序列類型並說明理由：
1. $2,5,8,11,\ldots$
2. $81,27,9,3,\ldots$
3. $1,1,2,3,5,\ldots$
4. $3,6,12,24,48,\ldots$

延伸：若資料有雜訊（例如某一項輸入錯），你要怎麼設計「近似判斷」？


In [13]:
def detect_type(arr):
    if len(arr) < 3:
        return "資料太少"

    diffs = [arr[i] - arr[i - 1] for i in range(1, len(arr))]
    if all(d == diffs[0] for d in diffs):
        return "等差"

    if all(arr[i - 1] != 0 for i in range(1, len(arr))):
        ratios = [arr[i] / arr[i - 1] for i in range(1, len(arr))]
        if all(r == ratios[0] for r in ratios):
            return "等比"

    return "其他"


print(detect_type([2, 5, 8, 11]))
print(detect_type([3, 6, 12, 24]))
print(detect_type([1, 1, 2, 3, 5]))


等差
等比
其他


## 9. 演算法例題：LeetCode 724 Find Pivot Index
題意：找一個最左邊索引 $i$，使得
- 左側總和 $\sum_{k=0}^{i-1} \text{nums}[k]$
- 右側總和 $\sum_{k=i+1}^{n-1} \text{nums}[k]$
相等。

觀念轉換（前綴和思維）：
- 設 $\text{total} = \sum \text{nums}$。
- 走訪到索引 $i$ 前，$\text{left}$ 表示左側總和。
- 右側總和可寫成 $\text{right} = \text{total} - \text{left} - \text{nums}[i]$。
- 若 $\text{left}=\text{right}$，$i$ 就是答案。

時間複雜度 $O(n)$，空間複雜度 $O(1)$。


### 練習（Pivot Index 變形）
1. 回傳「所有」pivot index（若有多個）。
2. 回傳最右邊 pivot index。
3. 改題：找 index 使左和與右和差的絕對值最小。


In [14]:
# TODO: 回傳所有 pivot index

def all_pivot_indices(nums):
    total = sum(nums)
    left = 0
    ans = []
    for i, x in enumerate(nums):
        if left == total - left - x:
            ans.append(i)
        left += x
    return ans

print(all_pivot_indices([0, 0, 0, 0]))  # [0,1,2,3]


[0, 1, 2, 3]


In [15]:
def pivot_index(nums):
    total = sum(nums)
    left = 0

    for i, x in enumerate(nums):
        right = total - left - x
        if left == right:
            return i
        left += x

    return -1


print(pivot_index([1, 7, 3, 6, 5, 6]))  # 3
print(pivot_index([1, 2, 3]))           # -1
print(pivot_index([2, 1, -1]))          # 0


3
-1
0


### 為什麼這題很適合 APCS
- 很像「區間和平衡點」的文字題。
- 若用雙層迴圈每次重算左右和會變 `O(n^2)`，容易超時。
- 把「左右和」改寫成 `total` 與 `left` 的關係，是常見優化套路。


In [16]:
# APCS 風格輸入版本：
# 第一行 n
# 第二行 n 個整數

def solve():
    n = int(input().strip())
    nums = list(map(int, input().split()))
    print(pivot_index(nums))

# 測試時取消註解
# solve()


## 10. Fix-It 改錯題（基礎）
每題都先「找錯 -> 說原因 -> 修正」，再用小測資驗證。


### 題 1：等差第 n 項（buggy）


In [ ]:
def arithmetic_nth(a1, d, n):
    return a1 + n * d


<details open><summary>Hint</summary>

1. 這份講義定義第 1 項是 $a_1$。
2. 先測 $n=1$，應該回傳 $a_1$。

</details>


### 題 1：修正版


In [ ]:
def arithmetic_nth(a1, d, n):
    return a1 + (n - 1) * d


### 題 2：等比前 n 項和（buggy）


In [ ]:
def geometric_sum(a1, r, n):
    return a1 * (1 - r**n) // (1 - r)


<details open><summary>Hint</summary>

1. 公式有條件：$r 
e 1$。
2. 先代入 $r=1$，分母會是 0。

</details>


### 題 2：修正版


In [ ]:
def geometric_sum(a1, r, n):
    if r == 1:
        return a1 * n
    return a1 * (1 - r**n) // (1 - r)


### 題 3：Fibonacci 邊界（buggy）


In [ ]:
def fib_iter(n):
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b


<details open><summary>Hint</summary>

1. 測 $n=0$、$n=1$。
2. 迴圈沒跑時，回傳值要符合定義。

</details>


### 題 3：修正版


In [ ]:
def fib_iter(n):
    if n == 0:
        return 0
    if n == 1:
        return 1
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b


### 題 4：前綴和區間查詢（buggy）


In [ ]:
def range_sum(prefix, l, r):
    return prefix[r] - prefix[l]


<details open><summary>Hint</summary>

1. 這裡 `prefix[0]=0`，且查詢是 1-based。
2. 區間 $[l,r]$ 要減的是 $l-1$。

</details>


### 題 4：修正版


In [ ]:
def range_sum(prefix, l, r):
    return prefix[r] - prefix[l - 1]


### 題 5：Pivot Index 更新順序（buggy）


In [ ]:
def pivot_index(nums):
    total = sum(nums)
    left = 0
    for i, x in enumerate(nums):
        left += x
        right = total - left - x
        if left == right:
            return i
    return -1


<details open><summary>Hint</summary>

1. `left` 代表「目前 $i$ 左邊」的和。
2. 應先比較，再把 `nums[i]` 加進 `left`。

</details>


### 題 5：修正版


In [ ]:
def pivot_index(nums):
    total = sum(nums)
    left = 0
    for i, x in enumerate(nums):
        right = total - left - x
        if left == right:
            return i
        left += x
    return -1


## 11. 綜合練習
### 練習 1
輸入 $a_1,d,n$，輸出等差數列前 $n$ 項與前 $n$ 項和。

Hint:
- 先用一個變數 `cur` 從 $a_1$ 開始遞增。
- 用 list 收集前 $n$ 項，最後 `join` 輸出。

### 練習 2
輸入 $a_1,r,n$，輸出等比數列第 $n$ 項。
若超過 $10^9$，輸出 `TOO LARGE`。

Hint:
- 可先迴圈乘 $n-1$ 次，避免一次次方看不清流程。
- 比較時用 `abs(value) > 10**9`（若你想兼容負值）。

### 練習 3
輸入 $n$，輸出 Fibonacci 第 $n$ 項。
要求時間複雜度 $O(n)$、空間 $O(1)$。

Hint:
- 保留兩個變數即可：前一項、目前項。
- 別忘記 $n=0,1$ 的特判。

### 練習 4
輸入一串數字與多次查詢 $[l,r]$，請用前綴和回答每次區間和。

Hint:
- 建 `prefix=[0]`，逐步 append 累計和。
- 查詢公式：`prefix[r]-prefix[l-1]`（1-based）。


## 12. 解題流程模板
1. 先判斷是「直接公式」還是「必須模擬」。
2. 寫出最小可行版本（先求對，再求快）。
3. 檢查邊界：`n=1`、`n=0`、負數、公比 `r=1`、可能溢位。
4. 若有多次查詢，思考是否需要前處理（例如前綴和）。
5. 最後才調整輸出格式。


In [17]:
# TODO: 在這格實作你自己的 APCS 數列題模板
# 建議先做：讀 input -> 建模 -> 輸出
pass
